### Transform curciuts table
1. Read bronze circuits table
2. keep only required columns
3. Use snake case for columns
4. Rename columns to more meaning full names
5. Filter out rows where circuitId is null
6. Remove duplicates
7. Use title case for circuit name and locality
8. Write transformed dataframe into silver circuits table

Below changes are required for incremental loading

1. Accept batch_id as param to notebook
2. Process data for only the batch_id
3. Add created, updated timestamp and batch_id to silver table
4. Merge the processed data to silber table

In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"

silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_df = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
    )

In [0]:
circuits_selected_df = circuits_df.select(
    F.col("circuitId").alias("circuit_id"),
    F.col("circuitName").alias("circuit_name"),
    F.col("lat").alias('latitude'),
    F.col("long").alias('longitude'),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

In [0]:
# Applying filters for null circuit_id

circuits_valid_df = circuits_selected_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
# Now remove the duplicate values
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
circuits_final_df = (
    circuits_distinct_df
    .withColumn("circuit_name", F.initcap("circuit_name"))
    .withColumn("locality", F.initcap("locality"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
    )

In [0]:
if not spark.catalog.tableExists(silver_table):
    (
    circuits_final_df.write
    .format('delta')
    .mode("overwrite")
    .saveAsTable(silver_table)
    )

else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            circuits_final_df.alias("s"),
            "t.circuit_id = s.circuit_id"
        )
        .whenMatchedUpdate(
            condition="s.batch_id >= t.batch_id",
            set={
                "circuit_name": "s.circuit_name",
                "latitude": "s.latitude",
                "longitude": "s.longitude",
                "locality": "s.locality",
                "country": "s.country",
                "ingestion_timestamp": "s.ingestion_timestamp",
                "source_file": "s.source_file",
                "batch_id": "s.batch_id",
                "updated_at": "s.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )